In [ ]:
!pip install dask-jobqueue dask-ml

In [ ]:
%matplotlib inline

In [1]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

# Configure to match your allocation settings
cluster = SLURMCluster( # LocalCluster, KubeCluster, KubernetesCluster
    queue='kura-wide',
    cores=5,
    memory='30GB',
    walltime='02:00:00',
    scheduler_options={
        'port': 8786,                # Hardcodes the communication channel
        'dashboard_address': ':8787' # Hardcodes the web UI channel
    }, 
)

# Scale to your 2 allocated nodes
cluster.scale(jobs=4)

client = Client(cluster)
print(client.dashboard_link)  # Click this to watch your cluster in real-time!

http://11.0.0.136:8787/status


In [2]:
from dask.distributed import Client

# Connect to your existing cluster
client = Client('tcp://11.0.0.136:8786')
client

<Client: 'tcp://11.0.0.136:8786' processes=20 threads=20, memory=111.80 GiB>

In [5]:
import dask
import dask.array as da
from dask_ml.datasets import make_classification

n, d = 100000, 100

X, y = make_classification(n_samples=n, n_features=d,
                           chunks=n // 10, flip_y=0.2)
X

dask.array<normal, shape=(100000, 100), dtype=float64, chunksize=(10000, 100), chunktype=numpy.ndarray>

In [6]:
from dask_ml.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y)
X_train

dask.array<concatenate, shape=(90000, 100), dtype=float64, chunksize=(9000, 100), chunktype=numpy.ndarray>

In [7]:
classes = da.unique(y_train).compute()
classes

array([0, 1])

In [8]:
from sklearn.linear_model import SGDClassifier

est = SGDClassifier(loss='log_loss', penalty='l2', tol=1e-3)

In [9]:
from dask_ml.wrappers import Incremental

inc = Incremental(est, scoring='accuracy')

In [10]:
inc.fit(X_train, y_train, classes=classes)

,estimator,SGDClassifier(loss='log_loss')
,scoring,'accuracy'
,shuffle_blocks,True
,random_state,None
,assume_equal_chunks,True
,predict_meta,None
,predict_proba_meta,None
,transform_meta,None
,loss,'log_loss'
,penalty,'l2'
,alpha,0.0001


In [11]:
inc.score(X_test, y_test)

np.float64(0.5848)

In [12]:
est = SGDClassifier(loss='log_loss', penalty='l2', tol=0e-3)
inc = Incremental(est, scoring='accuracy')

for _ in range(25):
    inc.partial_fit(X_train, y_train, classes=classes)
    print('Score:', inc.score(X_test, y_test))

Score: 0.5687
Score: 0.5728
Score: 0.5813
Score: 0.5967
Score: 0.5866
Score: 0.5941
Score: 0.5997
Score: 0.5972
Score: 0.612
Score: 0.6049
Score: 0.6146
Score: 0.6153
Score: 0.6095
Score: 0.6123
Score: 0.606
Score: 0.6111
Score: 0.6096
Score: 0.6111
Score: 0.6158
Score: 0.6192
Score: 0.618
Score: 0.6243
Score: 0.6168
Score: 0.6176
Score: 0.6225


In [13]:
inc.predict(X_test)  # Predict produces lazy dask arrays

dask.array<_predict, shape=(10000,), dtype=int64, chunksize=(1000,), chunktype=numpy.ndarray>

In [14]:
inc.predict(X_test)[:10000].compute()  # call compute to get results

array([0, 1, 1, ..., 0, 0, 0], shape=(10000,))

In [15]:
inc.score(X_test, y_test)

np.float64(0.6225)

In [16]:
import time
from dask.distributed import Client
from joblib import parallel_backend
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV  # Changed this

# 1. Start your Dask client (make sure this is initialized)
print(client)

# 2. Moderately sized dataset to make the parallelism worth it
X, y = make_classification(
    n_samples=5_000,  # Bumped up a bit
    n_features=50,
    n_informative=10,
    random_state=42
)

# Base model
model = RandomForestClassifier(random_state=42)

# 3. Define a hyperparameter search space
param_distributions = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "bootstrap": [True, False]
}

# 4. Set up the Search (this will generate 10 combos * 5 folds = 50 tasks)
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=10, 
    cv=5,
    n_jobs=-1,  # Tells scikit-learn to hand jobs over to the backend
    random_state=42
)

start = time.perf_counter()

# 5. Run it inside the Dask backend context
with parallel_backend("dask"):
    search.fit(X, y)

elapsed = time.perf_counter() - start

print(f"Best Score: {search.best_score_:.4f}")
print(f"Best Params: {search.best_params_}")
print(f"Elapsed HPO Time = {elapsed:.2f} seconds")

<Client: 'tcp://11.0.0.136:8786' processes=20 threads=20, memory=111.80 GiB>
Best Score: 0.9450
Best Params: {'n_estimators': 200, 'min_samples_split': 10, 'max_depth': 20, 'bootstrap': False}
Elapsed HPO Time = 15.44 seconds


In [17]:
import time
from dask.distributed import Client
from joblib import parallel_backend
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# =====================================================================
# 0. SETUP DATA AND SEARCH SPACE
# =====================================================================
X, y = make_classification(
    n_samples=5_000, 
    n_features=50,
    n_informative=10,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

param_distributions = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "bootstrap": [True, False]
}

# =====================================================================
# 1. NATIVE BENCHMARK (Standard scikit-learn Multi-core)
# =====================================================================
print("Starting Native Multi-core HPO...")
search_native = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=15, # Increased iterations slightly to give a better benchmark
    cv=5,
    n_jobs=-1, # Uses standard joblib multiprocessing
    random_state=42
)

start_native = time.perf_counter()
search_native.fit(X, y)
elapsed_native = time.perf_counter() - start_native
print(f"Native Local Time: {elapsed_native:.2f} seconds\n")

# =====================================================================
# 2. DASK LOCAL BENCHMARK
# =====================================================================
print("Spinning up local Dask cluster...")
# This automatically sets up a cluster using your local machine's resources
#client = Client(n_workers=4, threads_per_worker=2) 
print(f"Dask Dashboard available at: {client.dashboard_link}")

search_dask = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=15, 
    cv=5,
    n_jobs=-1, # Will be intercepted by the parallel_backend
    random_state=42
)

print("Starting Dask HPO...")
start_dask = time.perf_counter()
with parallel_backend("dask"):
    search_dask.fit(X, y)
elapsed_dask = time.perf_counter() - start_dask
print(f"Dask Local Time: {elapsed_dask:.2f} seconds\n")

# Close the local cluster cleanly
client.close()

# =====================================================================
# 3. FINAL COMPARISON
# =====================================================================
print("=== RESULTS ===")
print(f"Native Joblib: {elapsed_native:.2f}s")
print(f"Dask Backend:  {elapsed_dask:.2f}s")

difference = elapsed_native - elapsed_dask
if difference > 0:
    print(f"Dask was {difference:.2f}s faster!")
else:
    print(f"Native was {abs(difference):.2f}s faster (likely due to Dask local overhead).")

Starting Native Multi-core HPO...
Native Local Time: 34.16 seconds

Spinning up local Dask cluster...
Dask Dashboard available at: http://11.0.0.136:8787/status
Starting Dask HPO...
Dask Local Time: 22.10 seconds

=== RESULTS ===
Native Joblib: 34.16s
Dask Backend:  22.10s
Dask was 12.06s faster!
